# Lenguajes y Autómatas II (SCD-1016)
## Sesión 39 · Auditoría de Rendimiento y Fugas de Memoria en el Compilador

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo se demuestra que el compilador no se está comiendo la memoria ni volviéndose lento?
>
> **Objetivo de la sesión:** Medir dos síntomas de calidad del backend: cuántos bloques de arena consume un lote de objetos y cuántas reservas quedan sin liberar tras una compilación.

**Entregable:** tu implementación de bloques_arena() y fugas_en_log() en artesano_sesion39_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion39.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)


## 1. ¿Qué vas a construir?

Recibes un lote de objetos con su tamaño y el tamaño de bloque de la arena, o una bitácora de eventos m/f; devuelves cuántos bloques hacen falta o cuántas reservas quedaron vivas.

Al final de la sesión tendrás: **Un medidor de bloques de arena y un contador de fugas a partir de una bitácora de reservas y liberaciones.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 39](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion39.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion39.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion39.h"

int bloques_arena(int objetos, int tamano, int bloque);
int fugas_en_log(const char *log);
```

- Cada bloque de arena se llena antes de pedir el siguiente; se reserva el bloque completo aunque sobre espacio.
- La bitácora usa tokens separados por espacios: m reserva, f libera; cualquier otro token es inválido.
- Si hay más liberaciones que reservas o un token inválido, la bitácora es inconsistente y se responde -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion39_c.c` junto con `artesano_sesion39_c.c`
> (o `ia_sesion39_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `bloques_arena(1000, 32, 4096)` | 8 |
| 2 | `bloques_arena(10, 32, 1024)` | 1 |
| 3 | `bloques_arena(0, 32, 4096)` | 0 |
| 4 | `fugas_en_log("m f")` | 0 |
| 5 | `fugas_en_log("m m f")` | 1 |
| 6 | `fugas_en_log("f")` | -1 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 39 - Inspeccion de tiempos y fugas de memoria ===
Caso                               esperado:              obtenido:              resultado
[ 1] bloques_arena(1000, 32, 4096)      esperado: 8            obtenido: 8            OK
[ 2] bloques_arena(10, 32, 1024)        esperado: 1            obtenido: 1            OK
[ 3] bloques_arena(0, 32, 4096)         esperado: 0            obtenido: 0            OK
[ 4] fugas_en_log("m f")                esperado: 0            obtenido: 0            OK
[ 5] fugas_en_log("m m f")              esperado: 1            obtenido: 1            OK
[ 6] fugas_en_log("f")                  esperado: -1           obtenido: -1           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion39.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] bloques_arena redondea hacia arriba y no reserva bloques cuando no hay objetos.
- [ ] fugas_en_log distingue una bitácora equilibrada de una con reservas vivas o liberaciones de más.
- [ ] Los seis casos del harness salen OK y el programa termina con código de salida 0.


## Paso 0 · Entorno


In [ ]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion39_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [ ]:
%%writefile api_sesion39.h
/* api_sesion39.h - contrato de la Sesion 39
 * Inspección de tiempos y fugas de memoria
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION39_H
#define API_SESION39_H

int bloques_arena(int objetos, int tamano, int bloque);
int fugas_en_log(const char *log);

#endif /* API_SESION39_H */


In [ ]:
%%writefile artesano_sesion39_c.c
/* artesano_sesion39_c.c - Fase 1: El Artesano (sin IA)
 * Inspección de tiempos y fugas de memoria.
 *
 * Implementa lo que declara api_sesion39.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion39.h"
#include <stdio.h>
#include <string.h>

int bloques_arena(int objetos, int tamano, int bloque) {
    (void)objetos;
    (void)tamano;
    (void)bloque; /* TODO: cuantos bloques completos hacen falta */
    return 0;
}

int fugas_en_log(const char *log) {
    (void)log; /* TODO: contar m menos f, rechazando lo inconsistente */
    return 0;
}


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion39_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [ ]:
%%writefile ia_sesion39_c.c
/* ia_sesion39_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion39.h. No puede traer main().
 */
#include "api_sesion39.h"

/* TODO: pega aqui la propuesta de la IA */


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [ ]:
%%writefile harness_sesion39_c.c
/* harness_sesion39_c.c - auditoria automatica de la Sesion 39
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion39_c.c artesano_sesion39_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion39.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 39 - Inspeccion de tiempos y fugas de memoria ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int r = bloques_arena(1000, 32, 4096); verificar_int("bloques_arena(1000, 32, 4096)", (long)(8), (long)(r)); }
    { int r = bloques_arena(10, 32, 1024); verificar_int("bloques_arena(10, 32, 1024)", (long)(1), (long)(r)); }
    { int r = bloques_arena(0, 32, 4096); verificar_int("bloques_arena(0, 32, 4096)", (long)(0), (long)(r)); }
    { int r = fugas_en_log("m f"); verificar_int("fugas_en_log(\"m f\")", (long)(0), (long)(r)); }
    { int r = fugas_en_log("m m f"); verificar_int("fugas_en_log(\"m m f\")", (long)(1), (long)(r)); }
    { int r = fugas_en_log("f"); verificar_int("fugas_en_log(\"f\")", (long)(-1), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion39.md" "\n");
    return 0;
}


In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion39_c.c artesano_sesion39_c.c -o test_artesano && ./test_artesano

In [ ]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion39_c.c ia_sesion39_c.c -o test_ia && ./test_ia

## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion39.md` a `auditorias/auditoria_sesion39.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
